In [ ]:
# ==========================================================
# Import Required Library
# ==========================================================
# Import the pandas library for data loading and manipulation.

import pandas as pd

# ==========================================================
# Load the Original Dataset
# ==========================================================
# Read the full dataset from the specified file path.

main_data_path = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\02. data\Full data.csv"
df = pd.read_csv(main_data_path)

# ==========================================================
# Remove Unnecessary Columns
# ==========================================================
# Drop columns that are not required for the classification
# task to simplify the dataset.

columns_drop = [0, 1, 71, 72, 73, 74]
df.drop(df.columns[columns_drop], axis=1, inplace=True)

# Display the first few rows of the processed dataset.
print(df.head())

# ==========================================================
# Define Target Pictogram Classes
# ==========================================================
# Specify the GHS pictogram classes that will be extracted
# from the original dataset.

pictograms_to_keep = [
    "corrosive",
    "environmental hazard",
    "acute toxic",
    "health hazard",
    "flammable",
    "oxidizer",
    "irritant",
    "compressed gas",
    "explosive"
]

# ==========================================================
# Filter and Save Individual Pictogram Datasets
# ==========================================================
# Extract all samples belonging to each pictogram class,
# standardize the class label, and save each filtered
# dataset as a separate CSV file for subsequent model
# training and evaluation.

for pictogram in pictograms_to_keep:
    filtered_df = df[df["Pictogram(s)"].str.contains(pictogram, case=False, na=False)].copy()

    # Standardize the target label.
    filtered_df["Pictogram(s)"] = pictogram

    # Save the filtered dataset.
    save_path = rf"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data\filtered_{pictogram.replace(' ', '_')}.csv"
    filtered_df.to_csv(save_path, index=False)

    # Display the number of samples extracted for each class.
    print(f"filtered_{pictogram.replace(' ', '_')}.csv: {len(filtered_df)} rows")

In [ ]:
# ==========================================================
# Import Required Libraries
# ==========================================================
# Import libraries for data manipulation, preprocessing,
# model training, evaluation, and visualization.

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import os

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import KFold
from sklearn.multioutput import MultiOutputClassifier


# ==========================================================
# Helper Function
# ==========================================================
# Extract numerical values from strings containing units
# (e.g., "25 mg/L" → 25.0). Non-numeric values are converted
# to NaN.

def clean_numeric_with_units(x):
    if pd.isna(x):
        return np.nan
        
    if isinstance(x, (int, float)):
        return x
        
    match = re.search(r"-?\d+\.?\d*", str(x))
    
    if match:
        return float(match.group())
        
    return np.nan


# ==========================================================
# Load Small Balanced Datasets
# ==========================================================
# Load the filtered datasets corresponding to the small-sized
# pictogram classes.

filtered_oxidizer = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data\filtered_oxidizer.csv"
filtered_compressed = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data\filtered_compressed_gas.csv"
filtered_explosive = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data\filtered_explosive.csv"

oxidizer_df = pd.read_csv(filtered_oxidizer)
compressed_df = pd.read_csv(filtered_compressed)
explosive_df = pd.read_csv(filtered_explosive)

datasets = {
    "oxidizer": oxidizer_df,
    "compressed_gas": compressed_df,
    "explosive": explosive_df
}


# ==========================================================
# Configure Experiment Settings
# ==========================================================
# Define the sample size and cross-validation parameters.

sample_size = 130
n_splits = 3
n_repeats = 50

results = []


# ==========================================================
# Repeated Model Training and Evaluation
# ==========================================================
# Randomly sample balanced datasets, preprocess the features,
# train the classifier, and evaluate its performance using
# K-Fold cross-validation.

for repeat in range(n_repeats):
    sampled_dfs = []

    # Randomly sample an equal number of records from each class.
    for label, df in datasets.items():
        df_sampled = df.sample(n=sample_size,replace=False,random_state=None).copy()
        df_sampled["class_label"] = label
        sampled_dfs.append(df_sampled)

    # Combine and shuffle all sampled classes.
    final_df = pd.concat(sampled_dfs, ignore_index=True)
    
    # Shuffle the dataset before model training.
    final_df = final_df.sample(frac=1).reset_index(drop=True)

    # Separate features and target labels.
    features = final_df.drop(columns=["class_label"])
    y = pd.DataFrame()

    for class_name in datasets.keys():
        y[class_name] = (final_df["class_label"] == class_name).astype(int)

    # Convert numeric values stored as strings.
    for col in features.columns:
        if features[col].dtype == object:
            cleaned = features[col].apply(clean_numeric_with_units)

            if cleaned.notna().sum() > 0:
                features[col] = cleaned
                
    # Identify numerical and categorical features.
    numeric_cols = features.select_dtypes(include=["int64", "float64"]).columns
    categorical_cols = features.select_dtypes(include=["object"]).columns

    # Handle missing numerical values.
    if len(numeric_cols) > 0:
        num_imputer = SimpleImputer(strategy="median",keep_empty_features=True)
        numeric_array = num_imputer.fit_transform(features[numeric_cols])
        features[numeric_cols] = pd.DataFrame(numeric_array,columns=numeric_cols,index=features.index)

    # Handle missing categorical values.
    if len(categorical_cols) > 0:
        cat_imputer = SimpleImputer(strategy="most_frequent",keep_empty_features=True)
        categorical_array = cat_imputer.fit_transform(features[categorical_cols])
        features[categorical_cols] = pd.DataFrame(categorical_array,columns=categorical_cols,index=features.index)

    # Encode categorical variables.
    le_dict = {}

    for col in categorical_cols:
        le = LabelEncoder()
        features[col] = le.fit_transform(features[col])
        le_dict[col] = le

    # Perform 3-fold cross-validation.
    kf = KFold(n_splits=n_splits,shuffle=True)

    for fold_counter, (train_idx, test_idx) in enumerate(kf.split(features), start=1):
        X_train = features.iloc[train_idx]
        X_test = features.iloc[test_idx]
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        # Train the Multi-Output Random Forest classifier.
        clf = MultiOutputClassifier(RandomForestClassifier(n_estimators=100,random_state=42))
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        # Calculate performance metrics for each output class.
        for i, class_name in enumerate(datasets.keys()):
            class_acc = accuracy_score(y_test.iloc[:, i],y_pred[:, i])
            class_f1 = f1_score(y_test.iloc[:, i],y_pred[:, i],zero_division=0)
            results.append({"n": repeat + 1,"k": fold_counter,"output": class_name,"accuracy": class_acc,"f1score": class_f1})


# ==========================================================
# Save Classification Results
# ==========================================================
# Export the evaluation results for all repetitions and
# cross-validation folds.

results_df = pd.DataFrame(results)
results_df.to_csv(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data (S)\result_small.csv",index=False)
print("\nAll results saved!")


# ==========================================================
# Visualize Model Performance
# ==========================================================
# Generate bar charts showing the F1-score distribution for
# each output class.

f1score_df = pd.read_csv(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data (S)\result_small.csv")
save_dir = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data (S)\plots_f1(S)"
os.makedirs(save_dir, exist_ok=True)
outputs = f1score_df["output"].unique()

for output_name in outputs:
    data = f1score_df[f1score_df["output"] == output_name].reset_index(drop=True)
    plt.figure(figsize=(12, 4))
    plt.bar(range(len(data)),data["f1score"],color="skyblue")
    plt.title(f"Output: {output_name}")
    plt.ylabel("F1-score")
    plt.ylim(0, 1)
    plt.xticks([])
    save_path = os.path.join(save_dir,f"{output_name}_f1_plot_balanced_S.png")
    plt.savefig(save_path, bbox_inches="tight")
    plt.close()
print("All plots saved.")


# ==========================================================
# Calculate Average F1-score
# ==========================================================
# Compute the average F1-score for each output class.

f1_df = pd.read_csv(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data (S)\result_small.csv")

average_f1_df = (f1_df.groupby("output")["f1score"].mean().reset_index())

average_f1_df.rename(columns={"f1score": "average_f1score"},inplace=True)

average_f1_df.to_csv(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data (S)\average_f1score_small.csv",index=False)

print("Average F1-scores saved.")

# ==========================================================
# Calculate Average Accuracy
# ==========================================================
# Compute the average accuracy for each output class.

accuracy_df = pd.read_csv(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data (S)\result_small.csv")

average_accuracy_df = (accuracy_df.groupby("output")["accuracy"].mean().reset_index())

average_accuracy_df.rename(columns={"accuracy": "average_accuracy"},inplace=True)

average_accuracy_df.to_csv(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data (S)\average_accuracy_small.csv",index=False)

print("Average accuracy saved.")

In [ ]:
# ==========================================================
# Import Required Libraries
# ==========================================================
# Import libraries for data manipulation, preprocessing,
# model training, evaluation, and result visualization.

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import os

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import KFold
from sklearn.multioutput import MultiOutputClassifier


# ==========================================================
# Helper Function
# ==========================================================
# Extract numerical values from strings containing units
# (e.g., "50 mg/L" → 50.0). Non-numeric values are returned
# as NaN.

def clean_numeric_with_units(x):
    if pd.isna(x):
        return np.nan
        
    if isinstance(x, (int, float)):
        return x

    match = re.search(r"-?\d+\.?\d*", str(x))

    if match:
        return float(match.group())

    return np.nan


# ==========================================================
# Load Medium Balanced Datasets
# ==========================================================
# Load the filtered datasets corresponding to the medium-
# sized pictogram classes.

filtered_corrosive = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data\filtered_corrosive.csv"
filtered_environmental = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data\filtered_environmental_hazard.csv"
filtered_acute = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data\filtered_acute_toxic.csv"
filtered_health = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data\filtered_health_hazard.csv"
filtered_flammable = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data\filtered_flammable.csv"

corrosive_df = pd.read_csv(filtered_corrosive)
env_df = pd.read_csv(filtered_environmental)
acute_df = pd.read_csv(filtered_acute)
health_df = pd.read_csv(filtered_health)
flammable_df = pd.read_csv(filtered_flammable)

datasets = {
    "corrosive": corrosive_df,
    "environmental_hazard": env_df,
    "acute_toxic": acute_df,
    "health_hazard": health_df,
    "flammable": flammable_df
}


# ==========================================================
# Configure Experiment Settings
# ==========================================================
# Define the sample size and cross-validation parameters.

sample_size = 6400
n_splits = 3
n_repeats = 50

results = []


# ==========================================================
# Repeated Model Training and Evaluation
# ==========================================================
# Randomly sample a balanced dataset for each repetition,
# preprocess the data, train the classifier, and evaluate
# its performance using K-Fold cross-validation.

for repeat in range(n_repeats):
    sampled_dfs = []

    # Randomly sample an equal number of observations
    # from each pictogram class.
    for label, df in datasets.items():
        df_sampled = df.sample(n=sample_size,replace=False,random_state=None).copy()
        df_sampled["class_label"] = label
        sampled_dfs.append(df_sampled)

    # Combine all sampled classes into one balanced dataset.
    final_df = pd.concat(sampled_dfs, ignore_index=True)

    # Shuffle the dataset before model training.
    final_df = final_df.sample(frac=1).reset_index(drop=True)

    # Separate features and target labels.
    features = final_df.drop(columns=["class_label"])
    y = pd.DataFrame()

    for class_name in datasets.keys():
        y[class_name] = (final_df["class_label"] == class_name).astype(int)

    # Convert numerical values stored as strings.
    for col in features.columns:
        if features[col].dtype == object:
            cleaned = features[col].apply(clean_numeric_with_units)

            if cleaned.notna().sum() > 0:
                features[col] = cleaned

    # Identify numerical and categorical features.
    numeric_cols = features.select_dtypes(include=["int64", "float64"]).columns
    categorical_cols = features.select_dtypes(include=["object"]).columns

    # Replace missing numerical values using the median.
    if len(numeric_cols) > 0:
        num_imputer = SimpleImputer(strategy="median",keep_empty_features=True)
        numeric_array = num_imputer.fit_transform(features[numeric_cols])
        features[numeric_cols] = pd.DataFrame(numeric_array,columns=numeric_cols,index=features.index)

    # Replace missing categorical values using the most
    # frequent category.
    if len(categorical_cols) > 0:
        cat_imputer = SimpleImputer(strategy="most_frequent",keep_empty_features=True)
        categorical_array = cat_imputer.fit_transform(features[categorical_cols])
        features[categorical_cols] = pd.DataFrame(categorical_array,columns=categorical_cols,index=features.index)

    # Encode categorical variables into numerical labels.
    le_dict = {}

    for col in categorical_cols:
        le = LabelEncoder()
        features[col] = le.fit_transform(features[col])
        le_dict[col] = le

    # Perform 3-fold cross-validation.
    kf = KFold(n_splits=n_splits,shuffle=True)

    for fold_counter, (train_idx, test_idx) in enumerate(kf.split(features), start=1):
        X_train = features.iloc[train_idx]
        X_test = features.iloc[test_idx]
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        # Train the Multi-Output Random Forest classifier.
        clf = MultiOutputClassifier(RandomForestClassifier(n_estimators=100,random_state=42))
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        # Calculate Accuracy and F1-score for each pictogram
        # class.
        for i, class_name in enumerate(datasets.keys()):
            class_acc = accuracy_score(y_test.iloc[:, i],y_pred[:, i])
            class_f1 = f1_score(y_test.iloc[:, i],y_pred[:, i],zero_division=0)
            results.append({"n": repeat + 1,"k": fold_counter,"output": class_name,"accuracy": class_acc,"f1score": class_f1})


# ==========================================================
# Save Classification Results
# ==========================================================
# Save the Accuracy and F1-score obtained for every
# repetition and cross-validation fold.

results_df = pd.DataFrame(results)
results_df.to_csv(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data (M)\result_medium.csv",index=False)
print("All results saved!")


# ==========================================================
# Visualize Model Performance
# ==========================================================
# Generate bar charts showing the F1-score obtained for each
# pictogram class.

f1score_df = pd.read_csv(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data (M)\result_medium.csv")
save_dir = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data (M)\plots_f1(M)"
os.makedirs(save_dir, exist_ok=True)
outputs = f1score_df["output"].unique()

for output_name in outputs:
    data = f1score_df[f1score_df["output"] == output_name].reset_index(drop=True)
    plt.figure(figsize=(12,4))
    plt.bar(range(len(data)),data["f1score"],color="skyblue")
    plt.title(f"Output: {output_name}")
    plt.ylabel("F1-score")
    plt.ylim(0,1)
    plt.xticks([])
    save_path = os.path.join(save_dir,f"{output_name}_f1_plot_balanced_M.png")
    plt.savefig(save_path, bbox_inches="tight")
    plt.close()
print("All plots saved.")


# ==========================================================
# Calculate Average F1-score
# ==========================================================
# Compute the average F1-score for each output class.

f1_df = pd.read_csv(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data (M)\result_medium.csv")

average_f1_df = (f1_df.groupby("output")["f1score"].mean().reset_index())

average_f1_df.rename(columns={"f1score": "average_f1score"},inplace=True)

average_f1_df.to_csv(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data (M)\average_f1score_medium.csv",index=False)

print("Average F1-scores saved.")

# ==========================================================
# Calculate Average Accuracy
# ==========================================================
# Compute the average accuracy for each output class.

accuracy_df = pd.read_csv(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data (M)\result_medium.csv")

average_accuracy_df = (accuracy_df.groupby("output")["accuracy"].mean().reset_index())

average_accuracy_df.rename(columns={"accuracy": "average_accuracy"},inplace=True)

average_accuracy_df.to_csv(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data (M)\average_accuracy_medium.csv",index=False)

print("Average accuracy saved.")

In [ ]:
# ==========================================================
# Import Required Libraries
# ==========================================================
# Import libraries for data manipulation, preprocessing,
# model training, evaluation, and cross-validation.

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import os

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import RepeatedKFold


# ==========================================================
# Load Large Balanced Dataset
# ==========================================================
# Load the filtered dataset representing the large-sized
# pictogram class.

filtered_data = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data\filtered_irritant_data.csv"

df = pd.read_csv(filtered_data)


# ==========================================================
# Helper Function
# ==========================================================
# Extract numerical values from strings containing units
# (e.g., "25 mg/L" → 25.0). Non-numeric values are returned
# as NaN.

def clean_numeric_with_units(x):
    if pd.isna(x):
        return np.nan
        
    if isinstance(x, (int, float)):
        return x

    match = re.search(r"-?\d+\.?\d*", str(x))

    if match:
        return float(match.group())

    return np.nan


# ==========================================================
# Prepare Features and Target Labels
# ==========================================================
# Separate the predictor variables and target labels for
# model training.

output = df[["Pictogram(s)"]].values

feature_cols = [col for col in df.columns
    if col != "Pictogram(s)"]
features = df[feature_cols].copy()


# ==========================================================
# Data Preprocessing
# ==========================================================
# Convert numerical values stored as strings, handle missing
# values, and encode categorical variables.

for col in features.columns:
    if features[col].dtype == object:
        cleaned = features[col].apply(clean_numeric_with_units)

        if cleaned.notna().sum() > 0:
            features[col] = cleaned

numeric_cols = features.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = features.select_dtypes(include=["object"]).columns

# Replace missing numerical values using the median.
num_imputer = SimpleImputer(strategy="median")
features[numeric_cols] = num_imputer.fit_transform(features[numeric_cols])

# Replace missing categorical values using the most frequent
# category.
cat_imputer = SimpleImputer(strategy="most_frequent")
features[categorical_cols] = cat_imputer.fit_transform(features[categorical_cols])

# Encode categorical features into numerical labels.
le_dict = {}

for col in categorical_cols:
    features[col] = features[col].astype(str)
    le = LabelEncoder()
    features[col] = le.fit_transform(features[col])
    le_dict[col] = le


# ==========================================================
# Configure Cross-Validation
# ==========================================================
# Evaluate the model using Repeated K-Fold Cross Validation.

n_splits = 3
n_repeats = 50
rkf = RepeatedKFold(n_splits=n_splits,n_repeats=n_repeats,random_state=42)

results = []


# ==========================================================
# Train and Evaluate the Random Forest Classifier
# ==========================================================
# Train the classifier for each fold and record the Accuracy
# and F1-score.

for fold_counter, (train_idx, test_idx) in enumerate(rkf.split(features), start=1):
    X_train = features.iloc[train_idx]
    X_test = features.iloc[test_idx]
    y_train = output[train_idx]
    y_test = output[test_idx]
    k = fold_counter % n_splits

    if k == 0:
        k = n_splits
    n = (fold_counter - 1) // n_splits + 1

    clf = RandomForestClassifier(n_estimators=100,random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test,y_pred,zero_division=0)
    results.append({"k": k,"n": n,"output": "Irritant","accuracy": acc,"f1score": f1})


# ==========================================================
# Save Classification Results
# ==========================================================
# Save the evaluation results for every cross-validation
# fold.

results_df = pd.DataFrame(results)
result_csv_path = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data (L)\result_large.csv"
results_df.to_csv(result_csv_path,index=False)
print("Results saved!")


# ==========================================================
# Visualize Model Performance
# ==========================================================
# Generate a bar chart showing the F1-score obtained across
# all cross-validation folds.

f1score_df = pd.read_csv(result_csv_path)
save_dir = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data (L)\plots_f1(L)"
os.makedirs(save_dir, exist_ok=True)
plt.figure(figsize=(12, 4))
plt.bar(range(len(f1score_df)),f1score_df["f1score"],color="skyblue")
plt.title("Output: Irritant")
plt.ylabel("F1-score")
plt.ylim(0, 1)
plt.xticks([])
save_path = os.path.join(save_dir,"Irritant_f1_plot_balanced_L.png")
plt.savefig(save_path, bbox_inches="tight")
plt.close()
print("F1-score plot saved.")


# ==========================================================
# Calculate Average F1-score
# ==========================================================
# Compute the average F1-score for each output class.

f1_df = pd.read_csv(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data (L)\result_large.csv")

average_f1_df = (f1_df.groupby("output")["f1score"].mean().reset_index())

average_f1_df.rename(columns={"f1score": "average_f1score"},inplace=True)

average_f1_df.to_csv(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data (L)\average_f1score_large.csv",index=False)

print("Average F1-scores saved.")

# ==========================================================
# Calculate Average Accuracy
# ==========================================================
# Compute the average accuracy for each output class.

accuracy_df = pd.read_csv(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data (L)\result_large.csv")

average_accuracy_df = (accuracy_df.groupby("output")["accuracy"].mean().reset_index())

average_accuracy_df.rename(columns={"accuracy": "average_accuracy"},inplace=True)

average_accuracy_df.to_csv(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\07. multibinary\balanced data (L)\average_accuracy_large.csv",index=False)

print("Average accuracy saved.")

In [ ]:
# ==========================================================
# Import Required Libraries
# ==========================================================
# pandas: load and handle dataset
# numpy: numerical operations and index handling
# matplotlib: visualization of comparison results

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# ==========================================================
# Load Comparison Dataset
# ==========================================================
# This dataset contains F1-score results for both:
# - Imbalanced dataset model
# - Balanced dataset model

df = pd.read_csv(r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\08. analysation\imbalance balance.csv")


# ==========================================================
# Prepare Plot Axes
# ==========================================================
# x-axis represents different GHS output classes
# width controls spacing between grouped bars

x = np.arange(len(df["Output"]))
width = 0.35
fig, ax = plt.subplots(figsize=(15, 6))


# ==========================================================
# Extract Performance Values
# ==========================================================
# y_1 → F1-score from imbalanced dataset model
# y_2 → F1-score from balanced dataset model

y_1 = df["Imbalance"]
y_2 = df["Balance"]


# ==========================================================
# Plot Bar Chart Comparison
# ==========================================================
# Side-by-side bar chart is used to clearly compare
# performance between imbalanced and balanced datasets

ax.bar(x - width / 2,y_1,width=width,color="blue",label="Imbalance")
ax.bar(x + width / 2,y_2,width=width,color="orange",label="Balance")


# ==========================================================
# Chart Formatting
# ==========================================================
# Set axis labels and title for clarity

ax.set_xticks(x)
ax.set_xticklabels(df["Output"])
ax.set_xlabel("GHS Classes")
ax.set_ylabel("F1-score")
ax.set_title("Comparison of Imbalanced vs Balanced Dataset Performance")

# Add legend to distinguish both models
ax.legend()


# ==========================================================
# Save and Display Plot
# ==========================================================
# Save figure as high-resolution image for report use

plt.tight_layout()
plt.savefig("imvsbal.png",dpi=300,bbox_inches="tight")
plt.show()